In [6]:
import psycopg2
import os
import tempfile

def get_cert_file_from_env():
    cert_content = os.environ.get('IBM_DB_CERTIFICATE')
    if not cert_content:
        print("Warning: IBM_DB_CERTIFICATE not found in environment")
        return None
    
    # Replace \n with actual newlines
    cert_content = cert_content.replace('\\n', '\n')
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.crt', delete=False) as f:
        f.write(cert_content)
        return f.name

def test_connection():
    try:
        # Check environment variables
        user = os.environ.get('IBM_DB_USER')
        password = os.environ.get('IBM_DB_PASSWORD')
        
        if not user:
            print("Error: IBM_DB_USER not set")
            return
        if not password:
            print("Error: IBM_DB_PASSWORD not set")
            return
            
        cert_file = get_cert_file_from_env()
        
        # Use PUBLIC endpoint (remove .private)
        db_config = {
            'host': '77ffc5dd-8640-4646-a7a6-beb1dea99edb.bn2a2uid0up8mv7mv2ig.databases.appdomain.cloud',
            'port': 31173,
            'database': 'ibmclouddb',
            'user': user,
            'password': password,
            'sslmode': 'require',
            'connect_timeout': 30  # Increased timeout
        }
        
        # Add certificate if available
        if cert_file:
            db_config['sslrootcert'] = cert_file
        
        print(f"Attempting connection to: {db_config['host']}:{db_config['port']}")
        print(f"Database: {db_config['database']}")
        print(f"User: {db_config['user']}")
        print(f"SSL Certificate: {'Yes' if cert_file else 'No'}")
        
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Test the connection
        cursor.execute("SELECT version();")
        version = cursor.fetchone()[0]
        print(f"Connected successfully!")
        print(f"PostgreSQL version: {version}")
        
        cursor.close()
        conn.close()
        
    except psycopg2.OperationalError as e:
        if "timeout" in str(e).lower():
            print("Connection timeout - possible causes:")
            print("1. Using private endpoint instead of public")
            print("2. IP not whitelisted in IBM Cloud")
            print("3. Firewall blocking the connection")
        print(f"Connection Error: {e}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        # Clean up certificate file
        if 'cert_file' in locals() and cert_file:
            try:
                os.unlink(cert_file)
            except:
                pass

# Run the test
test_connection()

Attempting connection to: 77ffc5dd-8640-4646-a7a6-beb1dea99edb.bn2a2uid0up8mv7mv2ig.databases.appdomain.cloud:31173
Database: ibmclouddb
User: ibm_cloud_a0f965b9_f907_4e43_b759_7f77e30a3428
SSL Certificate: Yes
Connection timeout - possible causes:
1. Using private endpoint instead of public
2. IP not whitelisted in IBM Cloud
3. Firewall blocking the connection
Connection Error: connection to server at "77ffc5dd-8640-4646-a7a6-beb1dea99edb.bn2a2uid0up8mv7mv2ig.databases.appdomain.cloud" (169.62.134.98), port 31173 failed: timeout expired
connection to server at "77ffc5dd-8640-4646-a7a6-beb1dea99edb.bn2a2uid0up8mv7mv2ig.databases.appdomain.cloud" (169.61.150.218), port 31173 failed: timeout expired
connection to server at "77ffc5dd-8640-4646-a7a6-beb1dea99edb.bn2a2uid0up8mv7mv2ig.databases.appdomain.cloud" (169.48.161.197), port 31173 failed: timeout expired



In [ ]:


analysis_queries = [
    """
    -- Top 5 OEMs by total vehicles
    SELECT 
        oem,
        SUM(vehicle_count) as total_vehicles,
        COUNT(DISTINCT level_0_country) as countries,
        COUNT(DISTINCT body_type) as body_types
    FROM fact_registered_vehicles
    GROUP BY oem
    ORDER BY total_vehicles DESC
    LIMIT 5;
    """,
    
    """
    -- Market share by country and body type
    SELECT 
        level_0_country,
        body_type,
        COUNT(DISTINCT oem) as num_oems,
        SUM(total_vehicles_country) as total_vehicles,
        AVG(market_share_country) as avg_market_share
    FROM fact_market_share_country
    GROUP BY level_0_country, body_type
    ORDER BY total_vehicles DESC
    LIMIT 5;
    """,
    
    """
    -- Fuel type distribution
    SELECT 
        fuel_type,
        "Energy_Source",
        COUNT(DISTINCT oem) as num_oems,
        SUM(vehicle_count) as total_vehicles,
        COUNT(DISTINCT level_0_country) as num_countries
    FROM fact_registered_vehicles
    GROUP BY fuel_type, "Energy_Source"
    ORDER BY total_vehicles DESC
    LIMIT 5;
    """,
    
    """
    -- Regional market share analysis
    SELECT 
        level_0_country,
        level_1_region_name,
        COUNT(DISTINCT oem) as num_oems,
        AVG(market_share_district) as avg_market_share,
        SUM(total_vehicles_district) as total_vehicles
    FROM fact_market_share_district
    GROUP BY level_0_country, level_1_region_name
    ORDER BY total_vehicles DESC
    LIMIT 5;
    """
]


try: 
    # Run analysis queries
    print("\nRunning analysis queries...")
    for i, query in enumerate(analysis_queries, 1):
        print(f"\nAnalysis Query {i}:")
        print("-" * 80)
        cursor.execute(query)
        columns = [desc[0] for desc in cursor.description]
        print("Columns:", columns)
        print("\nResults:")
        results = cursor.fetchall()
        for row in results:
            print(row)
                
except Exception as e:
    print(f"Error: {str(e)}")
finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()

print("\nAnalysis completed.")